# Pipeline — DML na Camada Bronze (Delta Lake)

Executa operações de INSERT incremental, UPDATE, DELETE e MERGE (UPSERT) diretamente nas tabelas Delta Lake do bucket `bronze`.

## 1. Configuração

In [ ]:
MINIO_ENDPOINT   = "http://localhost:9010"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin"
BRONZE_BUCKET    = "s3a://bronze"

## 2. SparkSession com suporte a S3A e Delta Lake

In [ ]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("bronze_dml")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.hadoop.fs.s3a.endpoint",               MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key",             MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key",             MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access",      "true")
    .config("spark.hadoop.fs.s3a.impl",                   "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print(f"SparkSession iniciada — versão: {spark.version}")

## 3. Registro das tabelas no catálogo Spark

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp, col
from pyspark.sql.types import IntegerType, DoubleType

TABLES = ["funcionarios", "departamentos", "cargos", "folha_pagamento"]

for name in TABLES:
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS bronze_{name}
        USING DELTA
        LOCATION '{BRONZE_BUCKET}/{name}'
    """)

delta_funcionarios   = DeltaTable.forPath(spark, f"{BRONZE_BUCKET}/funcionarios")
delta_folha          = DeltaTable.forPath(spark, f"{BRONZE_BUCKET}/folha_pagamento")

print("Tabelas carregadas do catálogo.")

## 4. INSERT — Novos registros incrementais

Simula a chegada de novos funcionários provenientes do sistema de RH e insere via `append`.

In [ ]:
from pyspark.sql import Row
from datetime import date

# Novos registros chegando do sistema de origem
# Ajuste os campos conforme o schema real das suas tabelas no PostgreSQL
novos_funcionarios = [
    Row(
        funcionario_id=16,
        nome="Renata Gomes",
        email="renata.gomes@empresa.com",
        departamento_id=1,
        cargo_id=1,
        salario=5000.0,
        status="ATIVO",
        data_admissao=date(2024, 3, 1),
        source_system="sistema_rh",
    ),
    Row(
        funcionario_id=17,
        nome="Gustavo Prado",
        email="gustavo.prado@empresa.com",
        departamento_id=3,
        cargo_id=4,
        salario=10500.0,
        status="ATIVO",
        data_admissao=date(2024, 3, 15),
        source_system="sistema_rh",
    ),
]

df_novos = (
    spark.createDataFrame(novos_funcionarios)
    .withColumn("bronze_ingested_at", current_timestamp())
    .withColumn("funcionario_id",  col("funcionario_id").cast(IntegerType()))
    .withColumn("departamento_id", col("departamento_id").cast(IntegerType()))
    .withColumn("cargo_id",        col("cargo_id").cast(IntegerType()))
    .withColumn("salario",         col("salario").cast(DoubleType()))
)

df_novos.write.format("delta").mode("append").partitionBy("status").save(f"{BRONZE_BUCKET}/funcionarios")

total = spark.read.format("delta").load(f"{BRONZE_BUCKET}/funcionarios").count()
print(f"INSERT concluído. Total de funcionários: {total}")

## 5. UPDATE — Reajuste salarial via DeltaTable API

In [ ]:
# Reajuste de 10% para analistas sênior ativos (cargo_id = 3)
delta_funcionarios.update(
    condition="cargo_id = 3 AND status = 'ATIVO'",
    set={
        "salario":             "salario * 1.10",
        "bronze_ingested_at": "current_timestamp()",
    }
)

print("UPDATE (API) executado — reajuste de 10% para analistas sênior ativos.")
spark.sql("SELECT nome, cargo_id, salario, status FROM bronze_funcionarios WHERE cargo_id = 3").show()

In [ ]:
# UPDATE via SQL — desliga funcionários inativos com mais de 5 anos de casa
spark.sql("""
    UPDATE bronze_funcionarios
    SET    status             = 'DESLIGADO',
           bronze_ingested_at = current_timestamp()
    WHERE  status = 'INATIVO'
    AND    datediff(current_date(), data_admissao) > 1825
""")

print("UPDATE (SQL) executado — inativos há mais de 5 anos marcados como DESLIGADO.")
spark.sql("SELECT nome, status, data_admissao FROM bronze_funcionarios WHERE status = 'DESLIGADO'").show()

## 6. DELETE — Remoção seletiva de registros

In [ ]:
antes = spark.read.format("delta").load(f"{BRONZE_BUCKET}/funcionarios").count()
print(f"Registros antes do DELETE: {antes}")

# Remove definitivamente os funcionários com status DESLIGADO
delta_funcionarios.delete(condition="status = 'DESLIGADO'")

depois = spark.read.format("delta").load(f"{BRONZE_BUCKET}/funcionarios").count()
print(f"Registros após o DELETE:   {depois}")
print(f"{antes - depois} registro(s) removido(s).")

In [ ]:
# DELETE via SQL — remove competências antigas da folha de pagamento
spark.sql("""
    DELETE FROM bronze_folha_pagamento
    WHERE competencia < '2024-01'
""")

print("DELETE (SQL) executado — competências anteriores a 2024 removidas da folha.")

## 7. MERGE (UPSERT) — Sincronização incremental

Simula dados chegando de uma API de RH: atualiza registros existentes e insere novos.

In [ ]:
dados_incremental = [
    # Funcionário existente com salário atualizado
    Row(
        funcionario_id=1,
        nome="Ana Paula Silva",
        email="ana.silva@empresa.com",
        departamento_id=1,
        cargo_id=3,
        salario=9200.0,
        status="ATIVO",
        data_admissao=date(2018, 6, 1),
        source_system="api_rh",
    ),
    # Funcionário novo
    Row(
        funcionario_id=18,
        nome="Sofia Barros",
        email="sofia.barros@empresa.com",
        departamento_id=4,
        cargo_id=1,
        salario=4800.0,
        status="ATIVO",
        data_admissao=date(2024, 4, 1),
        source_system="api_rh",
    ),
]

df_incremental = (
    spark.createDataFrame(dados_incremental)
    .withColumn("bronze_ingested_at", current_timestamp())
)

(
    delta_funcionarios.alias("target")
    .merge(
        df_incremental.alias("source"),
        "target.funcionario_id = source.funcionario_id",
    )
    .whenMatchedUpdate(set={
        "salario":             "source.salario",
        "email":               "source.email",
        "bronze_ingested_at": "current_timestamp()",
    })
    .whenNotMatchedInsertAll()
    .execute()
)

total = spark.read.format("delta").load(f"{BRONZE_BUCKET}/funcionarios").count()
print(f"MERGE concluído. Total de funcionários: {total}")

## 8. Time Travel — Inspeção do histórico de versões

In [ ]:
print("=== Histórico de versões — bronze/funcionarios ===")
delta_funcionarios.history().select(
    "version", "timestamp", "operation", "operationMetrics"
).show(truncate=False)

In [ ]:
# Lê o snapshot da versão 0 (dados originais vindos do landing-zone)
df_v0 = (
    spark.read
    .format("delta")
    .option("versionAsOf", 0)
    .load(f"{BRONZE_BUCKET}/funcionarios")
)
print(f"Versão 0 (carga inicial): {df_v0.count()} registros")
df_v0.select("funcionario_id", "nome", "salario", "status").show(5)

In [ ]:
spark.stop()
print("SparkSession encerrada.")